# SSVEP EEG Classification with CCA

12-class SSVEP classification from 8-channel EEG using canonical correlation
analysis (CCA) reference features.

- **Data:** 10 subjects x 180 trials x 8 channels x 1024 samples (256 Hz)
- **Classes:** 12 stimulus frequencies, 9.25–14.75 Hz
- **Data source:** [YuDongPan/SSVEPNet](https://github.com/YuDongPan/SSVEPNet), `data/Dial/`
- **Method:** notch 50 Hz + band-pass 0.5–50 Hz, CCA against sine/cosine references with 3 harmonics, then standard CCA (argmax), logistic regression, and KNN
- **Evaluation:** 5-fold by trial and leave-one-subject-out, no trial IDs used as features

> **Note:** the earlier version of this notebook reported 100% accuracy. That number came from trial leakage: the `Trial` column was used as a feature, and random row splits let the same trial appear in train and test. This notebook fixes that.

In [ ]:
import json
from pathlib import Path

import numpy as np
from scipy.io import loadmat
from scipy.signal import butter, iirnotch, lfilter
from sklearn.cross_decomposition import CCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
TARGETS = [9.25, 11.25, 13.25, 9.75, 11.75, 13.75,
           10.25, 12.25, 14.25, 10.75, 12.75, 14.75]
FS = 256
TIME_STEPS = 1024
N_CHANNELS = 8
N_TRIALS = 180
N_SUBJECTS = 10
HARMONICS = 3

# Clone https://github.com/YuDongPan/SSVEPNet next to this notebook,
# or change this path.
DATA_DIR = Path("SSVEPNet/data/Dial")

In [ ]:
def bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyquist = 0.5 * fs
    b, a = butter(order, [lowcut / nyquist, highcut / nyquist], btype="band")
    return lfilter(b, a, data)


def notch_filter(data, notch_freq, fs, quality_factor=30):
    b, a = iirnotch(notch_freq, quality_factor, fs)
    return lfilter(b, a, data)


def preprocess_eeg(data, fs=FS, lowcut=0.5, highcut=50.0, notch_freq=50.0):
    data = notch_filter(data, notch_freq, fs)
    return bandpass_filter(data, lowcut, highcut, fs)


def reference_signals(freq, fs=FS, time_steps=TIME_STEPS, harmonics=HARMONICS):
    t = np.arange(time_steps) / fs
    cols = []
    for h in range(1, harmonics + 1):
        cols.append(np.sin(2 * np.pi * freq * h * t))
        cols.append(np.cos(2 * np.pi * freq * h * t))
    return np.stack(cols, axis=1)


def cca_score(trial, freq, fs=FS):
    """Canonical correlation between one EEG trial and a frequency reference."""
    x = trial.T  # time x channels
    y = reference_signals(freq, fs=fs)
    cca = CCA(n_components=1, scale=False, max_iter=1000)
    x_c, y_c = cca.fit_transform(x, y)
    if np.std(x_c[:, 0]) < 1e-12 or np.std(y_c[:, 0]) < 1e-12:
        return 0.0
    return abs(float(np.corrcoef(x_c[:, 0], y_c[:, 0])[0, 1]))

In [ ]:
def load_trials(data_dir: Path):
    signals, labels, subjects, trials = [], [], [], []
    for subject in range(1, N_SUBJECTS + 1):
        data = loadmat(data_dir / f"DataSub_{subject}.mat")["Data"]
        lab = loadmat(data_dir / f"LabSub_{subject}.mat")["Label"].ravel()
        for trial in range(N_TRIALS):
            signals.append(preprocess_eeg(data[:, :, trial]))
            labels.append(int(lab[trial]) - 1)
            subjects.append(subject)
            trials.append((subject - 1) * N_TRIALS + trial)
    return np.stack(signals), np.array(labels), np.array(subjects), np.array(trials)


def build_features(signals):
    features = []
    for i, trial in enumerate(signals):
        features.append([cca_score(trial, f) for f in TARGETS])
        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(signals)} trials")
    return np.array(features)


def evaluate_classifier(X, y, groups, splitter, clf):
    accs, f1s = [], []
    splits = splitter.split(X, y, groups) if isinstance(splitter, LeaveOneGroupOut) else splitter.split(X, y)
    for train_idx, test_idx in splits:
        clf.fit(X[train_idx], y[train_idx])
        pred = clf.predict(X[test_idx])
        accs.append(accuracy_score(y[test_idx], pred))
        f1s.append(f1_score(y[test_idx], pred, average="macro"))
    return {
        "accuracy_mean": float(np.mean(accs)),
        "accuracy_std": float(np.std(accs)),
        "macro_f1_mean": float(np.mean(f1s)),
        "macro_f1_std": float(np.std(f1s)),
        "per_fold_accuracy": [float(v) for v in accs],
    }

In [ ]:
signals, labels, subjects, trials = load_trials(DATA_DIR)
print(f"trials: {len(signals)} | subjects: {len(np.unique(subjects))} | classes: {len(np.unique(labels))}")

print("Computing CCA features (12 frequencies x 3 harmonics)...")
X = build_features(signals)

cca_pred = X.argmax(axis=1)
cca_acc = accuracy_score(labels, cca_pred)
subject_acc = [accuracy_score(labels[subjects == s], cca_pred[subjects == s]) for s in np.unique(subjects)]
print(f"\nStandard CCA: accuracy={cca_acc:.4f}, per-subject mean={np.mean(subject_acc):.4f} ± {np.std(subject_acc):.4f}")

In [ ]:
results = {
    "dataset": {
        "subjects": int(len(np.unique(subjects))),
        "trials": int(len(labels)),
        "classes": int(len(np.unique(labels))),
        "channels": N_CHANNELS,
        "sampling_rate_hz": FS,
        "time_steps": TIME_STEPS,
    },
    "features": {"targets_hz": TARGETS, "harmonics": HARMONICS},
    "standard_cca": {
        "accuracy": float(cca_acc),
        "per_subject_accuracy": [float(v) for v in subject_acc],
    },
}

classifiers = [
    ("logreg", make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))),
    ("knn_k3", make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3))),
]

for name, clf in classifiers:
    results[name] = {
        "5fold_by_trial": evaluate_classifier(
            X, labels, trials,
            StratifiedKFold(n_splits=5, shuffle=True, random_state=42), clf),
        "leave_one_subject_out": evaluate_classifier(
            X, labels, subjects, LeaveOneGroupOut(), clf),
    }
    print(f"\n{name}")
    print(f"  5-fold by trial: {results[name]['5fold_by_trial']['accuracy_mean']:.4f} ± {results[name]['5fold_by_trial']['accuracy_std']:.4f}")
    print(f"  leave-one-subject-out: {results[name]['leave_one_subject_out']['accuracy_mean']:.4f} ± {results[name]['leave_one_subject_out']['accuracy_std']:.4f}")

Path("results.json").write_text(json.dumps(results, indent=2))
print("\nSaved results.json")

## Results

| Model | 5-fold by trial | Leave-one-subject-out |
|---|---|---|
| Standard CCA (argmax) | 93.00% | 93.00% |
| CCA + Logistic Regression | 95.11 ± 0.89% | 94.28 ± 10.53% |
| CCA + KNN (k=3) | 93.56 ± 1.47% | 93.28 ± 10.80% |

Standard CCA is training-free, so its overall accuracy is the same under both partitions. Per-subject mean accuracy is 93.00 ± 11.86%: subjects 1–2 are much harder than the rest.

## Why the old 100% was invalid

The original notebook kept `Trial` as a feature and split rows randomly. Each trial has 8 channel rows, so the same trial could appear in both train and test. After removing the trial ID and splitting by trial, the old CCA + KNN pipeline drops to ~12.9% (chance for 12 classes is 8.3%). The 100% number must not be cited.

## Limitations

- Only 12 fixed frequencies from one dataset.
- 4-second trials; shorter windows and real-time use are not tested.
- Leave-one-subject-out variance is high (about 10% standard deviation).
- CCA assumes the stimulus frequencies are known in advance.
- Not a clinical or consumer BCI product.